In [1]:
import os
from google.colab import drive

# 2. 挂载 Google Drive
drive.mount('/content/drive')

# 3. 设置 Google Drive 里的文件夹路径
# 请确保你的文件放在了云盘的 Uppsala_Bot/knowledge 目录下
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/'
KNOWLEDGE_DIR = os.path.join(DRIVE_PROJECT_DIR, 'knowledge')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%pip install edge-tts
!pip install -U langchain-community langchain-core langchain-text-splitters

In [ ]:
import json
import os
import re
from threading import Thread
import time
import gradio as gr
import numpy as np
import pandas as pd
import requests
import torch
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    GenerationConfig,
    TextIteratorStreamer,
    logging,
)
import collections
import edge_tts
import asyncio

async def text_to_speech(text: str, output_file: str = "output.mp3", voice: str = "en-US-AvaNeural"):
    communicate = edge_tts.Communicate(text, voice)
    await communicate.save(output_file)
    return output_file

def generate_audio(text: str):
    # 只朗读前 250 个字符的核心结论，降低延迟
    short_text = text[:250] if len(text) > 250 else text
    asyncio.run(text_to_speech(short_text, "response_audio.mp3"))
    return "response_audio.mp3"

logging.set_verbosity_warning()

# =====================================================================
# 1. 数据准备与配置
# =====================================================================
try:
  df_hist = pd.read_csv(os.path.join(DRIVE_PROJECT_DIR, 'uppsala_tmax_1985_2026.csv'), parse_dates=['date'])
  print('Historical weather data loaded successfully! Shape:', df_hist.shape)
except Exception as e:
  print('Warning: CSV file not found. Creating a dummy dataset for demonstration.')
  dates = pd.date_range('1985-06-01', '2026-06-30')
  df_hist = pd.DataFrame({'date': dates, 'tmax': np.random.uniform(10, 32, len(dates))})

WEATHERMETRO_API_KEY = 'uXQyXC4iqxCVdzGSCbQ0uGmir0CzRFtU'
DEFAULT_LAT = 59.8586
DEFAULT_LON = 17.6389
DEFAULT_PLACE_NAME = 'Uppsala Centrum'
_weather_cache = {}

UPPSALA_PLACES_JSON = os.path.join(KNOWLEDGE_DIR, 'uppsala_all_places.json')
def load_uppsala_places():
  if os.path.exists(UPPSALA_PLACES_JSON):
    with open(UPPSALA_PLACES_JSON, 'r', encoding='utf-8') as f:
      return json.load(f)
  print('⚠️ 警告: 地名 JSON 文件不存在，将使用默认市中心坐标。')
  return []

UPPSALA_PLACES = load_uppsala_places()

# =====================================================================
# 2. 地名提取与 API 工具函数
# =====================================================================
def extract_location_coords(user_text: str):
  text_lower = user_text.lower()
  sorted_places = sorted(UPPSALA_PLACES, key=lambda x: len(x['name']), reverse=True)
  for place in sorted_places:
    name = place['name']
    if len(name) > 1 and name.lower() in text_lower:
      return {'found': True, 'name': name, 'lat': place['lat'], 'lon': place['lon'], 'type': place.get('type', 'landmark')}
  return {'found': False, 'name': DEFAULT_PLACE_NAME, 'lat': DEFAULT_LAT, 'lon': DEFAULT_LON, 'type': 'city_center'}

def detect_weather_intent(user_text: str):
    text = user_text.lower()
    forecast_keywords = ["tomorrow", "forecast", "next week", "future", "will it", "predict"]
    is_forecast = any(kw in text for kw in forecast_keywords)
    days_match = re.search(r'(\d+)\s*(?:days|day|天)', text)
    days = int(days_match.group(1)) if days_match else (3 if is_forecast else 1)
    return "forecast" if is_forecast else "current", days

def fetch_weathermetro_data(lat: float, lon: float, place_name: str = "Uppsala", user_text: str = "") -> dict:
    intent, forecast_days = detect_weather_intent(user_text)
    now = time.time()
    cache_key = f"{intent}_{round(lat, 3)}_{round(lon, 3)}"

    if cache_key in _weather_cache:
        cached_data, timestamp = _weather_cache[cache_key]
        if now - timestamp < 600:
            return cached_data

    headers = {"User-Agent": "UppsalaClimateBot/1.0"}
    url = "https://weathermetro.com/api/v2/forecast" if intent == "forecast" else "https://weathermetro.com/api/v2/current"
    params = {"api_key": WEATHERMETRO_API_KEY, "lat": lat, "lng": lon, "units": "metric"}
    if intent == "forecast":
        params["days"] = forecast_days

    try:
        response = requests.get(url, params=params, headers=headers, timeout=8)
        if response.status_code == 200:
            data = response.json()
            resp_data = data.get("response", {}).get("data", {})
            if intent == "forecast":
                hourly = resp_data.get("hourly", {})
                times, temps, precips = hourly.get("time", []), hourly.get("temperature_2m", []), hourly.get("precipitation", [])
                daily_grouped = collections.defaultdict(lambda: {"temps": [], "precips": []})
                for t, temp, precip in zip(times, temps, precips):
                    daily_grouped[t.split("T")[0]]["temps"].append(temp)
                    daily_grouped[t.split("T")[0]]["precips"].append(precip)
                parsed_daily = [
                    {"date": k, "temp_max": round(max(v["temps"]), 1), "temp_min": round(min(v["temps"]), 1), "precip_total": round(sum(v["precips"]), 1)}
                    for k, v in daily_grouped.items() if v["temps"]
                ]
                result = {"status": "success", "intent": "forecast", "place_name": place_name, "lat": lat, "lng": lon, "forecast_days": forecast_days, "daily": parsed_daily}
            else:
                current = data.get("response", {}).get("current", {})
                result = {"status": "success", "intent": "current", "place_name": place_name, "lat": lat, "lng": lon, "temp": current.get("temperature_2m", 7.4), "humidity": current.get("relative_humidity_2m", 80), "wind_speed": current.get("wind_speed_10m", 10), "condition": "Live Weather"}
            _weather_cache[cache_key] = (result, now)
            return result
    except Exception as e:
        print(f"WeatherMetro API Exception: {e}")

    return {"status": "fallback", "intent": intent, "place_name": place_name, "lat": lat, "lng": lon, "temp": 7.4, "condition": "Fallback Weather"}

def compute_heatwave_risk(days: int, temp: float, df_hist=df_hist):
  if not isinstance(days, (int, np.integer)) or days < 1 or not isinstance(temp, (int, float, np.floating)):
    return {'level': 'invalid'}
  record_max = float(df_hist['tmax'].max())
  if temp >= 30 and days >= 3:
    return {'level': 'level2', 'description': 'Extreme heat (Level 2): ≥30°C for ≥3 days', 'record_max': record_max}
  return {'level': 'none', 'description': 'Below defined heatwave thresholds', 'record_max': record_max}

# =====================================================================
# 3. 模型与 System Prompt 加载
# =====================================================================
MODEL_NAME = 'Qwen/Qwen3.5-2B'
embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
print(f'Loading model {MODEL_NAME}...')
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map='auto')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

temp_config = GenerationConfig(do_sample=False, max_new_tokens=1024, temperature=0.6)

# 增强版的 SYSTEM_PROMPT，严厉禁止事实捏造
SYSTEM_PROMPT = """You are CLIMES-Monitor, an intelligent weather, climate, and public-health assistant serving Uppsala, Sweden.

Your strict operational rules:
1. Grounding & Zero-Hallucination on Current Status:
   - Always check the provided [REALTIME STATUS EVALUATION] section in the user prompt.
   - If it states weather is NORMAL, NEVER claim or imply that Uppsala is currently under a heatwave or extreme weather event.
   - State the current actual temperature first, then provide official guidelines strictly as PRECAUTIONARY / HYPOTHETICAL advice (e.g., "Uppsala is currently not under a heatwave (today is 18°C), but according to official Swedish manuals, if extreme heat occurs...").

2. Direct & Helpful Advice:
   - Synthesize practical recommendations from RAG manuals into clear, friendly guidance.
   - Provide general public-health advice for heatwaves, including hydration, sun protection, and vulnerable populations if the user asks, but always preface with the current weather status.
   - Provide general public-health advice for extreme weather events (e.g., storms, floods) if the user asks, but always preface with the current weather status.
   - If the user query is on-topic, always provide a helpful answer, even if it's in other languages.
   - Some knowledge may be in other languages (e.g., Swedish). If relevant, translate or summarize key points into English for clarity.

3. Refusal Scope:
   - Refuse ONLY completely off-topic queries (e.g., coding troubleshooting, gaming, recipes) with: "I cannot provide information on topics unrelated to Uppsala weather, climate, or public health."
"""

def query_rag_knowledge(query_text: str, top_k: int = 3, score_threshold: float = 0.35) -> str:
  if not os.path.exists('./knowledge'):
    return ''
  vectorstore = Chroma(persist_directory='./knowledge', embedding_function=embeddings)
  results_with_scores = vectorstore.similarity_search_with_relevance_scores(query_text, k=top_k)
  context_chunks = []
  for doc, score in results_with_scores:
    if score >= score_threshold:
      source_name = os.path.basename(doc.metadata.get('source', 'Guide'))
      context_chunks.append(f"[{source_name}]: {doc.page_content}")
  return '\n\n'.join(context_chunks)

def extract_params(user_text):
  days_match = re.search(r'(\d+)\s*(?:天|days|day)', user_text, re.IGNORECASE)
  temp_match = re.search(r'(\d+(?:\.\d+)?)\s*(?:度|℃|°C|degrees)', user_text, re.IGNORECASE)
  days = int(days_match.group(1)) if days_match else None
  temp = float(temp_match.group(1)) if temp_match else None
  return days, temp

# =====================================================================
# 4. Gradio 核心响应逻辑（加入 Guardrail 规则判断）
# =====================================================================
def bot_response(message, history):
  if isinstance(message, str):
    user_text = message.strip()
  elif isinstance(message, dict):
    user_text = str(message.get("text", "")).strip()
  else:
    user_text = str(message).strip()

  loc_info = extract_location_coords(user_text)
  weather = fetch_weathermetro_data(
      lat=loc_info["lat"],
      lon=loc_info["lon"],
      place_name=loc_info["name"],
      user_text=user_text,
  )
  rag_context = query_rag_knowledge(user_text, top_k=3, score_threshold=0.35)

  days, temp_param = extract_params(user_text)
  eval_temp = temp_param if temp_param is not None else weather.get("temp")

  tool_result = None
  if days is not None and eval_temp is not None:
    try:
      tool_result = compute_heatwave_risk(
          days=int(days), temp=float(eval_temp)
      )
    except Exception as e:
      print("Tool execution error:", e)

  # -------------------------------------------------------------------------
  # Python 确定性防护栏 (Guardrail Calculation)
  # -------------------------------------------------------------------------
  current_temp = weather.get("temp", None)
  max_forecast_temp = None
  if weather.get("intent") == "forecast" and "daily" in weather:
    forecast_temps = [
        d.get("temp_max")
        for d in weather["daily"]
        if d.get("temp_max") is not None
    ]
    if forecast_temps:
      max_forecast_temp = max(forecast_temps)

  eval_temp_for_status = (
      max_forecast_temp if max_forecast_temp is not None else current_temp
  )

  if eval_temp_for_status is not None and eval_temp_for_status >= 30:
    extreme_weather_status = (
        "STATUS: EXTREME HEATWAVE WARNING ACTIVE (Predicted max:"
        f" {eval_temp_for_status}°C).\nINSTRUCTION: Treat heatwave safety"
        " advice as urgent, current instructions."
    )
  else:
    temp_disp = f"{current_temp}°C" if current_temp is not None else "normal"
    extreme_weather_status = (
        f"STATUS: NORMAL WEATHER (No active heatwave. Temperature is {temp_disp}).\n"
        "INSTRUCTION: Uppsala is NOT under a heatwave. If user asks for heatwave safety tips, "
        f"you MUST state that current weather is normal ({temp_disp}), and present safety tips strictly as PRECAUTIONARY advice."
    )

  # -------------------------------------------------------------------------
  # 组装完整的 Prompt 指令（包含湿度和风速）
  # -------------------------------------------------------------------------
  prompt_sections = [
      f"### REALTIME STATUS EVALUATION (GUARDRAIL)\n{extreme_weather_status}",
      f"### CURRENT USER QUERY\n{user_text}",
  ]

  if weather.get("intent") == "forecast" and "daily" in weather:
    forecast_str_list = [
        f"- {d.get('date')}: High {d.get('temp_max')}°C, Low {d.get('temp_min')}°C"
        for d in weather["daily"]
    ]
    prompt_sections.append(
        f"### WEATHER FORECAST DATA ({weather['place_name'].upper()})\n"
        + "\n".join(forecast_str_list)
    )
  else:
    # 🌟 重新拼入湿度与风速信息
    prompt_sections.append(
        f"### REAL-TIME WEATHER OBSERVATION ({weather['place_name'].upper()})\n"
        f"- Temp: {weather.get('temp', 'N/A')} °C | Condition:"
        f" {weather.get('condition', 'Live Weather')}\n"
        f"- Relative Humidity: {weather.get('humidity', 'N/A')}% | Wind Speed:"
        f" {weather.get('wind_speed', 'N/A')} km/h"
    )

  if rag_context:
    prompt_sections.append(
        f"### LOCAL MANUAL CONTEXT (RAG)\n```text\n{rag_context}\n```"
    )

  if tool_result and tool_result.get("level") != "invalid":
    prompt_sections.append(
        "### EXTREME HEATWAVE RISK METRICS\n- Risk Level:"
        f" {tool_result.get('level')}"
    )

  current_instruction = "\n\n".join(prompt_sections)

  conversation = [{"role": "system", "content": SYSTEM_PROMPT}]
  max_history_turns = 3
  recent_history = history[-max_history_turns:] if history else []

  for turn in recent_history:
    if isinstance(turn, dict):
      conversation.append({"role": turn["role"], "content": turn["content"]})

  conversation.append({"role": "user", "content": current_instruction})

  text_prompt = tokenizer.apply_chat_template(
      conversation, tokenize=False, add_generation_prompt=True
  )
  inputs = tokenizer([text_prompt], return_tensors="pt").to(model.device)

  streamer = TextIteratorStreamer(
      tokenizer, skip_prompt=True, skip_special_tokens=True
  )
  generation_kwargs = dict(
      **inputs, streamer=streamer, generation_config=temp_config
  )

  thread = Thread(target=model.generate, kwargs=generation_kwargs, daemon=True)
  thread.start()

  partial_text = ""
  for new_text in streamer:
    partial_text += new_text
    yield partial_text, None

  audio_path = generate_audio(partial_text)
  yield partial_text, audio_path

# =====================================================================
# 5. UI 交互界面
# =====================================================================
with gr.Blocks(title="CLIMES-Monitor: Uppsala Weather & Health") as demo:
  gr.Markdown("# 🌤️ CLIMES-Monitor: Uppsala Weather & Public-Health Agent")

  with gr.Row():
    with gr.Column(scale=4):
      chatbot = gr.Chatbot(label="Chat History", height=500)
      with gr.Row():
        user_input = gr.Textbox(placeholder="Ask about Uppsala weather or safety advice...", show_label=False, scale=6)
        submit_btn = gr.Button("Send 🚀", variant="primary", scale=1)
        stop_btn = gr.Button("Stop 🛑", variant="stop", scale=1)
      clear_btn = gr.Button("Clear History 🗑️")

    with gr.Column(scale=2):
      gr.Markdown("### 🔊 Voice Accessibility (TTS)")
      audio_player = gr.Audio(label="Audio Advice", autoplay=True, type="filepath")

  def user_submit(message, history):
    if not message.strip():
      return "", history
    history = history + [{"role": "user", "content": message}, {"role": "assistant", "content": ""}]
    return "", history

  def bot_streaming(history):
    user_message = history[-2]["content"]
    past_history = history[:-2]
    for text_chunk, audio_file in bot_response(user_message, past_history):
      history[-1]["content"] = text_chunk
      yield history, audio_file

  submit_event = user_input.submit(user_submit, [user_input, chatbot], [user_input, chatbot]).then(bot_streaming, [chatbot], [chatbot, audio_player])
  btn_event = submit_btn.click(user_submit, [user_input, chatbot], [user_input, chatbot]).then(bot_streaming, [chatbot], [chatbot, audio_player])
  stop_btn.click(fn=None, inputs=None, outputs=None, cancels=[submit_event, btn_event])
  clear_btn.click(lambda: ([], None), None, [chatbot, audio_player])

if __name__ == "__main__":
  demo.launch(share=True, debug=True)

/tmp/ipykernel_78406/1951700380.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


Historical weather data loaded successfully! Shape: (15004, 3)


/tmp/ipykernel_78406/1951700380.py:139: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading model Qwen/Qwen3.5-2B...


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://25540bbdd4309b5d8a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
Exception in thread Thread-9 (generate):
Traceback (most recent call last):
  

In [17]:
import requests

WEATHERMETRO_API_KEY = "uXQyXC4iqxCVdzGSCbQ0uGmir0CzRFtU"

# 1. 尝试调用 WeatherMetro 接口
url = "https://weathermetro.com/api/v2/current"  # 👈 确认你的 Endpoint 是否与官方文档完全一致
params = {
    "api_key": WEATHERMETRO_API_KEY,  # 👈 确认参数名是否是 apiKey (或者是 key / appid)
    "lat": 59.8586,
    "lng": 17.6389,
    "units": "metric",
}
headers = {"User-Agent": "UppsalaClimateBot/1.0"}

print("正在发起 WeatherMetro API 请求...")
response = requests.get(url, params=params, headers=headers, timeout=10)

print("HTTP 状态码:", response.status_code)
print("服务器返回内容:", response.text)
if response.status_code == 200:
    print("真实返回的 JSON 数据结构如下：")
    print(response.json())

正在发起 WeatherMetro API 请求...
HTTP 状态码: 200
服务器返回内容: {"meta":{"code":200,"version":"api.v2","terms":"https:\/\/weathermetro.com\/terms","attribution":"https:\/\/weathermetro.com\/attribution"},"response":{"current":{"time":"2026-10-14T21:00","interval":900,"temperature_2m":7.4,"relative_humidity_2m":89,"precipitation":0,"weather_code":null,"wind_speed_10m":12,"wind_direction_10m":309,"pressure_msl":1017.8,"cloud_cover":0},"units":{"time":"iso8601","interval":"seconds","temperature_2m":"\u00b0C","relative_humidity_2m":"%","precipitation":"mm","weather_code":"wmo code","wind_speed_10m":"km\/h","wind_direction_10m":"\u00b0","pressure_msl":"hPa","cloud_cover":"%"}}}
真实返回的 JSON 数据结构如下：
{'meta': {'code': 200, 'version': 'api.v2', 'terms': 'https://weathermetro.com/terms', 'attribution': 'https://weathermetro.com/attribution'}, 'response': {'current': {'time': '2026-10-14T21:00', 'interval': 900, 'temperature_2m': 7.4, 'relative_humidity_2m': 89, 'precipitation': 0, 'weather_code': None, 'wind_

In [27]:
import requests

WEATHERMETRO_API_KEY = "uXQyXC4iqxCVdzGSCbQ0uGmir0CzRFtU"
url = "https://weathermetro.com/api/v2/forecast"
params = {
    "api_key": WEATHERMETRO_API_KEY,
    "lat": 59.8586,
    "lng": 17.6389,
    "days": 3,
    "units": "metric",
}

res = requests.get(url, params=params)
if res.status_code == 200:
  print("真实预测 JSON 结构：")
  print(res.json())
else:
  print("报错状态码：", res.status_code, res.text)

真实预测 JSON 结构：
{'meta': {'code': 200, 'version': 'api.v2', 'terms': 'https://weathermetro.com/terms', 'attribution': 'https://weathermetro.com/attribution'}, 'response': {'data': {'latitude': 59.75, 'longitude': 17.75, 'generationtime_ms': 0.37980079650878906, 'utc_offset_seconds': 7200, 'timezone': 'Europe/Stockholm', 'timezone_abbreviation': 'GMT+2', 'hourly_units': {'time': 'iso8601', 'temperature_2m': '°C', 'precipitation': 'mm', 'weather_code': 'wmo code', 'wind_speed_10m': 'km/h'}, 'hourly': {'time': ['2026-10-14T00:00', '2026-10-14T01:00', '2026-10-14T02:00', '2026-10-14T03:00', '2026-10-14T04:00', '2026-10-14T05:00', '2026-10-14T06:00', '2026-10-14T07:00', '2026-10-14T08:00', '2026-10-14T09:00', '2026-10-14T10:00', '2026-10-14T11:00', '2026-10-14T12:00', '2026-10-14T13:00', '2026-10-14T14:00', '2026-10-14T15:00', '2026-10-14T16:00', '2026-10-14T17:00', '2026-10-14T18:00', '2026-10-14T19:00', '2026-10-14T20:00', '2026-10-14T21:00', '2026-10-14T22:00', '2026-10-14T23:00', '2026-10